In [ ]:
#Overview of Student's performance in Math subject
import pandas as pd
import zipfile
with zipfile.ZipFile('/content/archive.zip', 'r') as zip_ref:
    zip_ref.extractall('/content')
df = pd.read_csv('student_math_clean.csv')
df.head()

**Encoding**

In [ ]:
#Binary: Misc
df["school"] = df["school"].replace({"MS": 1, "GP": 0})
df["sex"] = df["sex"].replace({"F": 1, "M": 0})
df["address_type"] = df["address_type"].replace({"Urban": 1, "Rural": 0})
df["family_size"] = df["family_size"].replace({"Greater than 3": 1, "Less than or equal to 3": 0})
df["parent_status"] = df["parent_status"].replace({"Living together": 1, "Apart": 0})

#Binary: Yes/No
df["school_support"] = df["school_support"].replace({"yes": 1, "no": 0})
df["family_support"] = df["family_support"].replace({"yes": 1, "no": 0})
df["extra_paid_classes"] = df["extra_paid_classes"].replace({"yes": 1, "no": 0})
df["activities"] = df["activities"].replace({"yes": 1, "no": 0})
df["nursery_school"] = df["nursery_school"].replace({"yes": 1, "no": 0})
df["higher_ed"] = df["higher_ed"].replace({"yes": 1, "no": 0})
df["internet_access"] = df["internet_access"].replace({"yes": 1, "no": 0})
df["romantic_relationship"] = df["romantic_relationship"].replace({"yes": 1, "no": 0})

In [ ]:
#Ordinal
travel = {'<15 min.': 0, '15 to 30 min.': 1, '30 min. to 1 hour': 2, '>1 hour':3}
df["travel_time"] = df["travel_time"].map(travel)

study = {'<2 hours': 0, '2 to 5 hours': 1, '5 to 10 hours': 2, '>10 hours':3}
df["study_time"] = df["study_time"].map(study)

In [ ]:
#One-hot
categorical_cols = ["mother_education", "mother_job", "father_education", "father_job", "school_choice_reason", "guardian"]
encoded_df = pd.get_dummies(df, columns=categorical_cols, prefix=categorical_cols, prefix_sep='_')

In [ ]:
df["school_choice_reason"].value_counts()

In [ ]:
encoded_df

In [ ]:
encoded_df.info()

**Get X and y**

In [ ]:
y = encoded_df["final_grade"]
X = encoded_df.drop(["final_grade", "student_id", "school"], axis=1)

In [ ]:
y

**Train - test split**

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

**XGBoost model**

In [ ]:
import xgboost as xgb
from sklearn.metrics import mean_squared_error, r2_score

# Create and train the XGBoost model
xgb_model = xgb.XGBRegressor(
    alpha =5,
    colsample_bynode= 0.8,
    eta= 0.1,
    n_estimators= 100,
    subsample= 1)
# xgb_model = xgb.XGBRegressor(
#     colsample_bynode= 0.6,
#     eta= 0.05,
#     reg_lambda= 0.5,
#     n_estimators= 200,
#     subsample = 0.5) # You can adjust hyperparameters
xgb_model.fit(X_train, y_train)

# Make predictions
xgb_y_train_pred = xgb_model.predict(X_train)
xgb_y_test_pred = xgb_model.predict(X_test)

# Evaluate the XGBoost model
xgb_train_mse = mean_squared_error(y_train, xgb_y_train_pred)
xgb_train_r2 = r2_score(y_train, xgb_y_train_pred)
xgb_test_mse = mean_squared_error(y_test, xgb_y_test_pred)
xgb_test_r2 = r2_score(y_test, xgb_y_test_pred)

print(f"XGBoost Training MSE: {xgb_train_mse:.2f}")
print(f"XGBoost Training RMSE: {(xgb_train_mse)**0.5:.2f}")
print(f"XGBoost Training R-squared: {xgb_train_r2:.2f}")
print(f"XGBoost Testing MSE: {xgb_test_mse:.2f}")
print(f"XGBoost Testing RMSE: {(xgb_test_mse)**0.5:.2f}")
print(f"XGBoost Testing R-squared: {xgb_test_r2:.2f}")

In [ ]:
import shap
explainer = shap.Explainer(xgb_model)
#shap_values = explainer(X_test)

In [ ]:
shap.summary_plot(shap_values, X_test)

In [ ]:
shap.summary_plot(shap_values, X_test, plot_type="bar")

In [ ]:
explainer = shap.Explainer(xgb_model)
shap_values = explainer(X)
shap.plots.waterfall(shap_values[0])